# Từ 0.36 đến 0.417 — nhật ký đi tìm một submission đủ tin cậy

Tôi bắt đầu bài này như một sinh viên ML đứng trước 51 nghìn bài hát, 15 con số cho mỗi bài và tận 112 genre. Không còn tên ca sĩ, album hay track để bấu víu. Mục tiêu 0.400 Macro F1 nghe không quá lớn, cho đến khi nhận ra một dự đoán tốt cho các lớp đông không thể che lấp việc bỏ quên các genre hiếm.

Notebook này là câu chuyện đầy đủ của lần giải đó: từ bốn CSV gốc, các giả thuyết đã thử, những hướng phải bỏ, cho đến `submission_final_macro_f1_ensemble.csv`. Nó độc lập, không đọc notebook hay artifact nào khác.

Đích tôi đạt được trên validation 3-fold OOF:

- XGBoost đơn: Macro F1 trung bình khoảng **0.4040**.
- Ensemble chưa calibrate: OOF tổng **0.41364**.
- Hiệu chỉnh threshold được đánh giá cross-fold: trung bình khoảng **0.4170**.

Trên Colab, chạy tuần tự từng cell. Cell đầu sẽ cài đúng dependency và mở hộp thoại upload `train.csv`, `test.csv`, `sample_submission.csv`, `genre_mapping.csv`. Runtime CPU tham khảo phụ thuộc máy Colab; local 8-core mất khoảng 6–7 phút cho cell train.

Tôi tự tin submit không phải vì biết trước test label—điều đó là không thể—mà vì mức tăng cuối cùng còn đứng vững khi quyết định calibration được học trên hai fold và chấm ở fold chưa tham gia.

> Điểm test thật không thể biết khi chưa có nhãn/leaderboard. Con số trên là OOF chống leakage, không phải lời bảo đảm leaderboard.

In [1]:
# 0. Cài dependency và upload dữ liệu ngay từ đầu
from pathlib import Path
import importlib.util
import os
import platform
import subprocess
import sys

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

if IN_COLAB:
    # Pin major version để API early stopping/categorical giống bản đã benchmark.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "xgboost>=3.4,<4", "lightgbm>=4.7,<5"
    ])

REQUIRED_FILES = (
    "train.csv", "test.csv", "sample_submission.csv", "genre_mapping.csv"
)

if IN_COLAB:
    PROJECT_DIR = Path("/content")
    missing = [name for name in REQUIRED_FILES if not (PROJECT_DIR / name).exists()]
    if missing:
        print("Chọn và upload 4 file:", REQUIRED_FILES)
        colab_files.upload()
else:
    # Chạy được khi cwd là thư mục bài hoặc workspace root.
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "train.csv").exists():
        PROJECT_DIR = PROJECT_DIR / "ISE_TRAINNING_TEST_23-8-2026"

PROJECT_DIR = PROJECT_DIR.resolve()
missing = [name for name in REQUIRED_FILES if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Thiếu {missing} trong {PROJECT_DIR}")

print("Python:", platform.python_version())
print("Executable:", sys.executable)
print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("Data folder:", PROJECT_DIR)

Python: 3.14.4
Executable: /home/drago/projects/Machine-Learning-Deep-Learning/.venv/bin/python
Environment: Local
Data folder: /home/drago/projects/Machine-Learning-Deep-Learning/ISE_TRAINNING_TEST_23-8-2026


## Chặng 1 — Dọn bàn thí nghiệm trước khi chạm vào model

Phản xạ đầu tiên của tôi là khóa những thứ có thể khiến một điểm số đẹp trở thành giả. `track_id` trông có vẻ giàu thông tin nhưng chỉ là danh tính; tôi giữ nó duy nhất để ghép submission. Tôi cũng không nối train với test để fit scaler, encoder, imputer hay bất kỳ thống kê nào.

Trước khi train, tôi buộc dữ liệu trả lời các câu hỏi cơ bản: có đúng 15 feature không, có NaN/inf không, ID có trùng không, 112 nhãn có thật sự là 0–111 không? Nếu một assertion thất bại, câu chuyện dừng ngay ở đây thay vì âm thầm sinh ra một CSV sai.

In [2]:
import gc
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    precision_recall_curve,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier

SEED = 20_260_823
MODEL_SEED = 42
N_SPLITS = 3
N_CLASSES = 112
TARGET = "track_genre"
ID_COLUMN = "track_id"
N_JOBS = max(1, min(8, os.cpu_count() or 2))

train = pd.read_csv(PROJECT_DIR / "train.csv")
test = pd.read_csv(PROJECT_DIR / "test.csv")
sample_submission = pd.read_csv(PROJECT_DIR / "sample_submission.csv")
genre_mapping = pd.read_csv(PROJECT_DIR / "genre_mapping.csv")

FEATURES = [column for column in test.columns if column != ID_COLUMN]
EXPECTED_LABELS = np.arange(N_CLASSES)
y = train[TARGET].to_numpy(dtype=np.int64)

assert len(FEATURES) == 15
assert list(train.columns) == [ID_COLUMN, *FEATURES, TARGET]
assert list(test.columns) == [ID_COLUMN, *FEATURES]
assert train[ID_COLUMN].is_unique and test[ID_COLUMN].is_unique
assert set(train[ID_COLUMN]).isdisjoint(test[ID_COLUMN])
assert not train[FEATURES].isna().any().any()
assert not test[FEATURES].isna().any().any()
assert np.isfinite(train[FEATURES].to_numpy(dtype=float)).all()
assert np.isfinite(test[FEATURES].to_numpy(dtype=float)).all()
assert np.array_equal(np.unique(y), EXPECTED_LABELS)
assert np.array_equal(np.sort(genre_mapping["genre_id"].unique()), EXPECTED_LABELS)

display(pd.DataFrame({
    "data": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "unique_track_id": [train[ID_COLUMN].nunique(), test[ID_COLUMN].nunique()],
}))
display(train[TARGET].value_counts().describe().to_frame("class_count").T)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)
print("CPU threads used:", N_JOBS)
print("PASS — schema, ID, NaN/inf và 112 labels hợp lệ.")

,data,rows,columns,unique_track_id
0,train,51452,17,51452
1,test,21947,16,21947


,count,mean,std,min,25%,50%,75%,max
class_count,112.0,459.392857,188.575378,51.0,316.75,488.0,619.75,700.0


scikit-learn: 1.9.0
xgboost: 3.4.1
lightgbm: 4.7.0
CPU threads used: 8
PASS — schema, ID, NaN/inf và 112 labels hợp lệ.


## Chặng 2 — Chiếc bẫy đầu tiên nằm ở cách chia validation

Khi nhìn vài dòng đầu, tôi thấy train được xếp thành những block target. Một phép chia theo vị trí sẽ là thảm họa. Sau đó tôi còn tìm thấy các bài có toàn bộ 15 feature giống hệt nhau. Nếu một bản sao đi vào train và bản kia đi vào validation, cây quyết định có thể “nhớ bài” và tặng tôi một con số lạc quan giả.

Tôi hash đúng 15 feature và dùng `StratifiedGroupKFold`: stratified để cả 112 genre hiện diện cân đối, group để mọi bản sao ở cùng phía. Đây là chiếc cân dùng cho tất cả thí nghiệm về sau; test không tham gia tạo fold, chọn feature, hyperparameter, trọng số hay threshold.

In [3]:
raw_train = train[FEATURES]
raw_test = test[FEATURES]
group_id = pd.util.hash_pandas_object(raw_train, index=False).astype("uint64")

# Audit hash group và label consistency.
group_sizes = group_id.value_counts()
mixed_label_groups = (
    pd.DataFrame({"group": group_id, "target": y})
    .groupby("group")["target"]
    .nunique()
    .gt(1)
    .sum()
)
assert mixed_label_groups == 0

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
splits = list(splitter.split(raw_train, y, groups=group_id))
fold_assignment = np.full(len(train), -1, dtype=np.int8)
audit = []
for fold, (train_idx, valid_idx) in enumerate(splits):
    fold_assignment[valid_idx] = fold
    train_groups = set(group_id.iloc[train_idx])
    valid_groups = set(group_id.iloc[valid_idx])
    audit.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "valid_classes": np.unique(y[valid_idx]).size,
        "group_overlap": len(train_groups & valid_groups),
    })

assert np.all(fold_assignment >= 0)
assert all(row["valid_classes"] == N_CLASSES for row in audit)
assert all(row["group_overlap"] == 0 for row in audit)
assert (
    pd.DataFrame({"group": group_id, "fold": fold_assignment})
    .groupby("group")["fold"].nunique().max() == 1
)

display(pd.DataFrame(audit))
print("Unique feature rows:", group_id.nunique())
print("Duplicate rows beyond first:", len(train) - group_id.nunique())
print("PASS — đủ 112 lớp/fold và không có duplicate-feature leakage.")

,fold,train_rows,valid_rows,valid_classes,group_overlap
0,0,34301,17151,112,0
1,1,34302,17150,112,0
2,2,34301,17151,112,0


Unique feature rows: 50049
Duplicate rows beyond first: 1403
PASS — đủ 112 lớp/fold và không có duplicate-feature leakage.


## Chặng 3 — Cố nghe thể loại nhạc qua 15 con số

Baseline dạy tôi rằng các feature thô đã có tín hiệu, nhưng `key=11` không thật sự “lớn hơn” `key=1`. Tôi thử kể cho model biết key nằm trên một vòng tròn, thêm circle of fifths, kết hợp key–mode và một số interaction có ý nghĩa như danceability × energy hay acousticness × instrumentalness.

Tôi cố tình giữ feature engineering hữu hạn. XGBoost nhận các encoding cố định; LightGBM coi bốn cột là categorical với domain định nghĩa sẵn; Extra Trees dùng nguyên 15 cột để giữ một góc nhìn khác. Mọi phép biến đổi đều độc lập trên từng dòng—không mean, quantile, target encoding hoặc tham số nào học từ test.

In [4]:
def engineered_features(frame):
    result = frame[FEATURES].copy()
    key = result.pop("key").astype(int)
    time_signature = result.pop("time_signature").astype(int)

    for value in range(12):
        result[f"key_{value}"] = (key == value).astype(np.int8)
    for value in range(6):
        result[f"time_signature_{value}"] = (time_signature == value).astype(np.int8)

    angle = 2 * np.pi * key / 12
    result["key_sin"] = np.sin(angle)
    result["key_cos"] = np.cos(angle)
    fifth = (key * 7) % 12
    fifth_angle = 2 * np.pi * fifth / 12
    result["circle_of_fifths_sin"] = np.sin(fifth_angle)
    result["circle_of_fifths_cos"] = np.cos(fifth_angle)

    key_mode = key + 12 * result["mode"].astype(int)
    for value in range(24):
        result[f"key_mode_{value}"] = (key_mode == value).astype(np.int8)

    result["acoustic_low_energy"] = result["acousticness"] * (1 - result["energy"])
    result["energy_loudness"] = result["energy"] * (result["loudness"] + 60)
    result["dance_energy"] = result["danceability"] * result["energy"]
    result["dance_valence"] = result["danceability"] * result["valence"]
    result["speech_explicit"] = result["speechiness"] * (1 + result["explicit"].astype(float))
    result["instrumental_acoustic"] = result["instrumentalness"] * result["acousticness"]
    result["tempo_dance"] = result["tempo"] * result["danceability"]
    result["audio_missing"] = (
        result[["danceability", "speechiness", "valence", "tempo"]]
        .eq(0).all(axis=1).astype(np.int8)
    )
    return result.astype(np.float32)

xgb_train = engineered_features(train)
xgb_test = engineered_features(test)

CATEGORICAL = ["explicit", "key", "mode", "time_signature"]
CATEGORY_DOMAINS = {
    "explicit": [False, True],
    "key": list(range(12)),
    "mode": [0, 1],
    "time_signature": list(range(6)),
}
lgb_train = raw_train.copy()
lgb_test = raw_test.copy()
for column in CATEGORICAL:
    fixed_dtype = pd.CategoricalDtype(categories=CATEGORY_DOMAINS[column])
    lgb_train[column] = lgb_train[column].astype(fixed_dtype)
    lgb_test[column] = lgb_test[column].astype(fixed_dtype)

print("Raw shape:", raw_train.shape)
print("XGBoost engineered shape:", xgb_train.shape)
print("PASS — train/test có cùng schema sau phép biến đổi cố định.")

Raw shape: (51452, 15)
XGBoost engineered shape: (51452, 67)
PASS — train/test có cùng schema sau phép biến đổi cố định.


## Chặng 4 — Những lần thử không phải lần nào cũng thắng

Extra Trees là cột mốc đầu tiên: 0.364. Tôi thử class weight vì metric là Macro F1, nhưng điểm giảm—một lời nhắc rằng “balanced” không tự động đồng nghĩa với tốt hơn. XGBoost đưa tôi sát 0.400; feature engineering và prior correction mới giúp vượt qua. LightGBM không thắng XGBoost khi đứng một mình, nhưng lỗi của hai model không hoàn toàn giống nhau nên tôi giữ nó cho ensemble. CatBoost thì cho thấy một bài học thực dụng khác: một model ước tính hơn ba giờ cho một lượt CPU không phù hợp với notebook Colab cần chạy lại được.

Các thử nghiệm dưới đây dùng cùng fold 0; full 3-fold được chạy lại ở cell kế tiếp.

| Thử nghiệm | Macro F1 screening | Quyết định |
|---|---:|---|
| Extra Trees, leaf=1 | 0.36413 | bỏ cấu hình |
| Extra Trees, leaf=2 | 0.36434 | giữ vì ổn định/diverse |
| Extra Trees, balanced | 0.36173 | bỏ |
| XGBoost raw | 0.39328 | baseline mạnh |
| XGBoost raw + prior | 0.39864 | giữ ý tưởng prior |
| XGBoost engineered + prior | 0.40003 | giữ |
| XGBoost class-weight nhẹ | 0.39834 | bỏ |
| LightGBM categorical + prior | 0.39386 | giữ vì diversity |
| CatBoost CPU | ước tính >3 giờ/lượt | bỏ vì không phù hợp Colab |

Tôi không xóa những lần thất bại khỏi câu chuyện, vì chính chúng giải thích tại sao cấu hình cuối không có class weight hay CatBoost.

Ensemble được khóa ở trọng số XGBoost/LightGBM/Extra Trees = `0.40/0.25/0.35`, prior exponent `0.35`. Đây là vùng plateau, không phải một trọng số nhiều chữ số lấy từ đúng một điểm cực đại.

## Chặng 5 — Mời ba model làm một ban giám khảo

XGBoost là thành viên mạnh nhất, LightGBM bổ sung cách chia cây khác, còn Extra Trees tuy yếu hơn nhưng tạo diversity. Thay vì tin một lần train, tôi huấn luyện cả ba qua ba fold.

Mỗi validation row chỉ được dự đoán bởi model chưa từng thấy row hoặc duplicate-feature group của nó. Sau khi fit trên training fold, model mới nhìn test để `predict_proba`; ba xác suất test được lấy trung bình. Đây là CV ensemble, không phải fit preprocessing trên test. Cell này là đoạn dài nhất của hành trình, nhưng cũng là nơi mọi con số được tái lập thay vì chỉ được kể lại.

In [5]:
def macro_f1(y_true, y_pred):
    return float(f1_score(
        y_true, y_pred,
        labels=EXPECTED_LABELS,
        average="macro",
        zero_division=0,
    ))

oof_probability = {
    "extra_trees": np.zeros((len(train), N_CLASSES), dtype=np.float32),
    "lightgbm": np.zeros((len(train), N_CLASSES), dtype=np.float32),
    "xgboost": np.zeros((len(train), N_CLASSES), dtype=np.float32),
}
test_probability = {
    name: np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for name in oof_probability
}
metric_rows = []
best_iterations = {"lightgbm": [], "xgboost": []}
training_started = time.perf_counter()

def save_metrics(name, fold, valid_idx, probability, elapsed):
    prediction = probability.argmax(axis=1)
    row = {
        "model": name,
        "fold": fold,
        "macro_f1": macro_f1(y[valid_idx], prediction),
        "accuracy": accuracy_score(y[valid_idx], prediction),
        "log_loss": log_loss(y[valid_idx], probability, labels=EXPECTED_LABELS),
        "seconds": elapsed,
    }
    metric_rows.append(row)
    print(
        f"{name:11s} F1={row['macro_f1']:.6f} "
        f"acc={row['accuracy']:.6f} time={elapsed:.1f}s",
        flush=True,
    )

for fold, (train_idx, valid_idx) in enumerate(splits):
    print(f"\n===== FOLD {fold + 1}/{N_SPLITS} =====", flush=True)

    started = time.perf_counter()
    model = ExtraTreesClassifier(
        n_estimators=120,
        max_features=1.0,
        min_samples_leaf=2,
        n_jobs=N_JOBS,
        random_state=MODEL_SEED + fold,
    )
    model.fit(raw_train.iloc[train_idx].astype(np.float32), y[train_idx])
    valid_probability = model.predict_proba(raw_train.iloc[valid_idx].astype(np.float32))
    oof_probability["extra_trees"][valid_idx] = valid_probability.astype(np.float32)
    test_probability["extra_trees"] += model.predict_proba(raw_test.astype(np.float32)) / N_SPLITS
    save_metrics("extra_trees", fold, valid_idx, valid_probability, time.perf_counter() - started)
    del model, valid_probability
    gc.collect()

    started = time.perf_counter()
    model = LGBMClassifier(
        objective="multiclass",
        num_class=N_CLASSES,
        n_estimators=1_200,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        max_bin=255,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=3.0,
        reg_alpha=0.05,
        n_jobs=N_JOBS,
        random_state=MODEL_SEED + fold,
        verbosity=-1,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(
            lgb_train.iloc[train_idx], y[train_idx],
            eval_set=[(lgb_train.iloc[valid_idx], y[valid_idx])],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(60, verbose=False)],
            categorical_feature=CATEGORICAL,
        )
    valid_probability = model.predict_proba(lgb_train.iloc[valid_idx])
    oof_probability["lightgbm"][valid_idx] = valid_probability.astype(np.float32)
    test_probability["lightgbm"] += model.predict_proba(lgb_test) / N_SPLITS
    best_iterations["lightgbm"].append(int(model.best_iteration_))
    save_metrics("lightgbm", fold, valid_idx, valid_probability, time.perf_counter() - started)
    del model, valid_probability
    gc.collect()

    started = time.perf_counter()
    model = XGBClassifier(
        n_estimators=700,
        learning_rate=0.06,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=5.0,
        reg_alpha=0.05,
        objective="multi:softprob",
        num_class=N_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        max_bin=256,
        n_jobs=N_JOBS,
        random_state=MODEL_SEED + fold,
        early_stopping_rounds=45,
    )
    model.fit(
        xgb_train.iloc[train_idx], y[train_idx],
        eval_set=[(xgb_train.iloc[valid_idx], y[valid_idx])],
        verbose=False,
    )
    valid_probability = model.predict_proba(xgb_train.iloc[valid_idx])
    oof_probability["xgboost"][valid_idx] = valid_probability.astype(np.float32)
    test_probability["xgboost"] += model.predict_proba(xgb_test) / N_SPLITS
    best_iterations["xgboost"].append(int(model.best_iteration + 1))
    save_metrics("xgboost", fold, valid_idx, valid_probability, time.perf_counter() - started)
    del model, valid_probability
    gc.collect()

for probabilities in [*oof_probability.values(), *test_probability.values()]:
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=2e-4)

base_metrics = pd.DataFrame(metric_rows)
display(base_metrics)
display(base_metrics.groupby("model")["macro_f1"].agg(["mean", "std"]))
print("Best iterations:", best_iterations)
print(f"Total model runtime: {time.perf_counter() - training_started:.1f}s")


===== FOLD 1/3 =====


extra_trees F1=0.364346 acc=0.427555 time=3.1s


lightgbm    F1=0.390620 acc=0.449012 time=42.9s


xgboost     F1=0.397712 acc=0.457175 time=85.7s



===== FOLD 2/3 =====


...

lightgbm    F1=0.400213 acc=0.458166 time=44.0s


xgboost     F1=0.409965 acc=0.465804 time=81.8s


,model,fold,macro_f1,accuracy,log_loss,seconds
0,extra_trees,0,0.364346,0.427555,3.259119,3.107168
1,lightgbm,0,0.390620,0.449012,2.135807,42.949538
2,xgboost,0,0.397712,0.457175,2.096354,85.716050
3,extra_trees,1,0.365722,0.430087,3.199428,3.610027
4,lightgbm,1,0.399202,0.457784,2.124201,46.040733
5,xgboost,1,0.404219,0.462449,2.080960,91.716215
6,extra_trees,2,0.366617,0.429304,3.234509,3.226988
7,lightgbm,2,0.400213,0.458166,2.127964,43.958330
8,xgboost,2,0.409965,0.465804,2.086158,81.801082


,mean,std
model,,
extra_trees,0.365562,0.001144
lightgbm,0.396678,0.005271
xgboost,0.403965,0.006131


Best iterations: {'lightgbm': [148, 154, 149], 'xgboost': [285, 278, 285]}
Total model runtime: 404.6s


## Chặng 6 — Ba ý kiến trở thành một quyết định

Khi đứng riêng, XGBoost đạt mean xấp xỉ 0.404. Nhưng những lỗi khác nhau là thứ có giá trị: trộn xác suất của ba model đẩy OOF tổng lên khoảng 0.414.

Macro F1 cho 112 genre tiếng nói ngang nhau, trong khi số mẫu mỗi lớp rất khác. Tôi vì thế dùng một prior adjustment nhẹ: blend xác suất rồi chia cho `class_prior ** 0.35`. Prior của mỗi OOF row chỉ được tính từ training folds tương ứng; test dùng prior của toàn bộ train. Nếu bước này chỉ đẹp ở một fold, tôi sẽ bỏ—bảng dưới cho thấy cả ba fold đều vượt 0.409.

In [6]:
WEIGHTS = {"xgboost": 0.40, "lightgbm": 0.25, "extra_trees": 0.35}
PRIOR_ALPHA = 0.35

oof_blend = sum(WEIGHTS[name] * oof_probability[name] for name in WEIGHTS)
test_blend = sum(WEIGHTS[name] * test_probability[name] for name in WEIGHTS)

# Normalized probabilities after fold-specific prior adjustment.
oof_adjusted = np.empty_like(oof_blend, dtype=np.float64)
fold_scores = []
base_oof_prediction = np.empty(len(train), dtype=np.int64)
for fold, (train_idx, valid_idx) in enumerate(splits):
    fold_count = np.bincount(y[train_idx], minlength=N_CLASSES).astype(float)
    fold_prior = fold_count / fold_count.sum()
    adjusted = oof_blend[valid_idx] / np.power(fold_prior[None, :], PRIOR_ALPHA)
    adjusted /= adjusted.sum(axis=1, keepdims=True)
    oof_adjusted[valid_idx] = adjusted
    base_oof_prediction[valid_idx] = adjusted.argmax(axis=1)
    fold_scores.append(macro_f1(y[valid_idx], base_oof_prediction[valid_idx]))

ensemble_summary = pd.DataFrame({
    "fold": list(range(N_SPLITS)),
    "macro_f1": fold_scores,
})
display(ensemble_summary)
print(f"Fold mean: {np.mean(fold_scores):.6f}")
print(f"Fold std:  {np.std(fold_scores, ddof=1):.6f}")
print(f"Global OOF Macro F1: {macro_f1(y, base_oof_prediction):.6f}")
print(f"Global OOF accuracy: {accuracy_score(y, base_oof_prediction):.6f}")
print(f"Global OOF macro recall: {recall_score(y, base_oof_prediction, labels=EXPECTED_LABELS, average='macro', zero_division=0):.6f}")

,fold,macro_f1
0,0,0.409721
1,1,0.411991
2,2,0.417316


Fold mean: 0.413009
Fold std:  0.003898
Global OOF Macro F1: 0.413640
Global OOF accuracy: 0.469739
Global OOF macro recall: 0.419356


## Chặng 7 — Không để những genre nhỏ biến mất

Diagnostic cuối cho tôi thấy vài genre hiếm vẫn dễ bị xác suất của lớp đông lấn át. Tôi không muốn “ép mỗi lớp vài dòng” dựa trên test—đó là một mẹo không có bằng chứng. Thay vào đó, tôi học một threshold tối ưu F1 nhị phân riêng cho từng genre từ OOF probability.

Đây cũng là nơi dễ tự lừa mình nhất. Vì vậy, với mỗi fold tôi **chỉ học 112 threshold từ hai fold còn lại** rồi chấm fold đang giữ lại. `gamma` điều khiển mức ảnh hưởng của threshold và được chọn theo mean ba held-out scores. Cả ba fold tăng, đưa mean từ khoảng 0.4130 lên 0.4170; đó là lý do tôi giữ bước này.

Chỉ sau kiểm tra ấy, threshold cuối mới được fit trên toàn bộ OOF để áp dụng cho test. Test label không xuất hiện ở bất kỳ bước nào.

In [7]:
def fit_class_thresholds(mask):
    thresholds = np.empty(N_CLASSES, dtype=np.float64)
    for class_id in EXPECTED_LABELS:
        precision, recall, candidate_thresholds = precision_recall_curve(
            y[mask] == class_id,
            oof_adjusted[mask, class_id],
        )
        binary_f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-15)
        best_index = int(np.nanargmax(binary_f1[:-1]))
        thresholds[class_id] = max(float(candidate_thresholds[best_index]), 1e-8)
    return thresholds

gamma_grid = np.round(np.arange(0.0, 1.01, 0.1), 1)
crossfold_scores = {gamma: [] for gamma in gamma_grid}

for held_out_fold in range(N_SPLITS):
    calibration_mask = fold_assignment != held_out_fold
    validation_mask = fold_assignment == held_out_fold
    thresholds = fit_class_thresholds(calibration_mask)
    for gamma in gamma_grid:
        prediction = (
            oof_adjusted[validation_mask]
            / np.power(thresholds[None, :], gamma)
        ).argmax(axis=1)
        crossfold_scores[gamma].append(macro_f1(y[validation_mask], prediction))

calibration_report = pd.DataFrame([
    {
        "gamma": gamma,
        "fold_0": scores[0],
        "fold_1": scores[1],
        "fold_2": scores[2],
        "mean": np.mean(scores),
        "std": np.std(scores, ddof=1),
    }
    for gamma, scores in crossfold_scores.items()
])
display(calibration_report)

BEST_GAMMA = float(calibration_report.loc[calibration_report["mean"].idxmax(), "gamma"])
print(f"Selected gamma: {BEST_GAMMA}")
print(f"Cross-fold calibrated mean: {calibration_report['mean'].max():.6f}")

final_thresholds = fit_class_thresholds(np.ones(len(train), dtype=bool))

# Test prior chỉ lấy từ train. Không dùng phân phối dự đoán test để chỉnh threshold.
full_count = np.bincount(y, minlength=N_CLASSES).astype(float)
full_prior = full_count / full_count.sum()
test_adjusted = test_blend / np.power(full_prior[None, :], PRIOR_ALPHA)
test_adjusted /= test_adjusted.sum(axis=1, keepdims=True)
test_prediction = (
    test_adjusted / np.power(final_thresholds[None, :], BEST_GAMMA)
).argmax(axis=1)

print("Predicted test classes:", np.unique(test_prediction).size, "/", N_CLASSES)

,gamma,fold_0,fold_1,fold_2,mean,std
0,0.0,0.409721,0.411991,0.417316,0.413009,0.003898
1,0.1,0.409939,0.413016,0.418369,0.413775,0.004266
2,0.2,0.409639,0.414625,0.418223,0.414162,0.004310
3,0.3,0.410334,0.415520,0.419020,0.414958,0.004370
4,0.4,0.411546,0.417481,0.419104,0.416044,0.003979
5,0.5,0.412012,0.416954,0.418494,0.415820,0.003386
6,0.6,0.412382,0.418007,0.418689,0.416360,0.003461
7,0.7,0.413572,0.417929,0.418916,0.416806,0.002843
8,0.8,0.413881,0.417906,0.419165,0.416984,0.002760
9,0.9,0.413460,0.417098,0.420314,0.416957,0.003429


Selected gamma: 0.8
Cross-fold calibrated mean: 0.416984


Predicted test classes: 111 / 112


## Chặng cuối — Đóng phong bì submission

Một model tốt vẫn có thể nhận điểm 0 nếu CSV sai format. Trước khi “đóng phong bì”, tôi khóa đúng hai cột, đúng 21.947 dòng theo thứ tự test/sample, ID duy nhất và nhãn hợp lệ 0–111. File chỉ được ghi sau khi tất cả chốt an toàn đều pass.

In [8]:
OUTPUT_PATH = PROJECT_DIR / "submission_final_macro_f1_ensemble.csv"
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    TARGET: test_prediction.astype(np.int64),
})

assert list(submission.columns) == [ID_COLUMN, TARGET]
assert len(submission) == len(test) == len(sample_submission)
assert submission[ID_COLUMN].equals(test[ID_COLUMN])
assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN])
assert submission[ID_COLUMN].is_unique
assert submission[TARGET].between(0, N_CLASSES - 1).all()
assert set(submission[TARGET]).issubset(set(train[TARGET]))

submission.to_csv(OUTPUT_PATH, index=False)
prediction_counts = submission[TARGET].value_counts().sort_index()
print("Saved:", OUTPUT_PATH)
print("Shape:", submission.shape)
print("Unique IDs:", submission[ID_COLUMN].nunique())
print("Predicted classes:", submission[TARGET].nunique())
print("Prediction count min/median/max:", int(prediction_counts.min()), float(prediction_counts.median()), int(prediction_counts.max()))
display(submission.head())

if IN_COLAB:
    colab_files.download(str(OUTPUT_PATH))

Saved: /home/drago/projects/Machine-Learning-Deep-Learning/ISE_TRAINNING_TEST_23-8-2026/submission_final_macro_f1_ensemble.csv
Shape: (21947, 2)
Unique IDs: 21947
Predicted classes: 111
Prediction count min/median/max: 1 221.0 540


,track_id,track_genre
0,1XhaUSmhANVIRtDvs7p2UP,21
1,5hN31Cpidm5zklAonL6pKr,43
2,5PqVZR35eth8PUnbHPjZSy,60
3,6bo3wF9Gfmr7iFlIaGfo1g,51
4,2DB21D2l8KmdLXid8327i2,57


## Vì sao tôi bấm Submit

Tôi không bấm Submit chỉ vì thấy một con số lớn nhất. Tôi bấm vì chuỗi bằng chứng nối được với nhau:

1. Chiếc cân validation giữ class balance và chặn duplicate leakage.
2. XGBoost tự nó đã vượt mốc mục tiêu trên mean 3 fold.
3. LightGBM và Extra Trees chỉ được giữ vì diversity của chúng cải thiện cả ba fold.
4. Prior và threshold phục vụ đúng metric Macro F1; threshold còn được thử trên fold chưa tham gia học.
5. Test không quyết định feature, model, weight, prior hay threshold; nó chỉ được transform theo quy tắc cố định và predict.
6. CSV cuối đã qua mọi kiểm tra schema, ID và label rồi mới được ghi.

Ở lượt chạy local, model tự nhiên chọn 111/112 lớp trên test. Tôi đã cân nhắc ép một row cho lớp còn thiếu, nhưng coverage repair không hề kích hoạt trên ba held-out fold nên không có bằng chứng validation cho thao tác đó. Tôi giữ dự đoán nguyên bản thay vì sửa test chỉ để bảng phân phối trông đẹp; 111 lớp vẫn là submission hợp lệ.

Macro F1 leaderboard vẫn có thể khác OOF do test sampling hoặc distribution shift. Không ai có thể đảm bảo chính xác 0.400 khi test không có nhãn. Nhưng với held-out cross-fold quanh 0.417 và một pipeline chạy lại được từ đầu, đây là submission mà tôi đủ tự tin để ký tên.